# 06 — Bin 3 Embeddings via Cohere Direct API

**Purpose.** Embed the last outstanding slice of the corpus — **Bin 3 (report years 2022-2025)** —
using Cohere's **own API** instead of AWS Bedrock, then save it as an **independent parquet**
that gets merged into the main vectors table as a final, deliberate step.

**Why not Bedrock.** Bedrock's Cohere Embed V4 daily cap on account `mjsushanth_mlops` is
**8,100,000 tokens/day and is not adjustable via any API**. Bin 3 needs ~7.16M tokens, which
does not fit alongside anything else in the rolling 24h window — it blocked Bin 2 twice.
Cohere's direct API runs **the same model at the same $0.12/1M** with no daily cap.

**Why this is safe.** Notebook `05_CohereDirect_VectorSpaceValidation` established empirically
that Bedrock and Cohere-direct vectors are interchangeable: cosine **mean 0.99985** on 32
already-embedded Bin 1 sentences vs **0.2652** for control pairs, with 32/32 top-1
self-retrieval. The residual gap is Cohere's own run-to-run nondeterminism, not transport drift.

| | |
|---|---|
| Sentences to embed | **180,848** |
| Tokens | **7,162,360** |
| Cost @ $0.12/1M | **~$0.86** |
| API calls @ 96/batch | **1,884** |
| Rate ceiling | 2,000 inputs/min (Cohere docs) |
| Expected runtime | **~100 min** at a 0.9 safety factor |
| Key | **production** (trial is capped at 1,000 calls/month — cannot do this) |

### Design notes, deliberately simple
This is a **one-off script in notebook form**, not a reusable module. It happily duplicates
logic from `platform_core/embedding_generation.py` rather than importing or abstracting it.

Two places where it *improves* on the original, for good reason:

1. **Shard checkpoints instead of full-file overwrite.** The original rewrites one growing
   parquet every 50 batches. Here that file reaches ~700 MB, so ~37 rewrites would mean
   roughly 13 GB of pointless disk writes. Instead each checkpoint appends a small numbered
   shard, and the shards are concatenated at the end. Same resumability, a fraction of the I/O.
2. **Fatal-vs-transient error split.** Carried over from the Bedrock hard-stop fix: a quota or
   billing failure aborts immediately instead of burning the full backoff ladder against a wall
   that cannot move. Ordinary throttles still retry.

### Before running
- **Run headless and keep the Mac awake** for a ~100 minute job:
  `caffeinate -i jupyter execute 06_Bin3_CohereDirect_Embeddings.ipynb`
- The run cell is **idempotent and resumable** — re-running skips whatever the shards already hold.
- The final merge cell is **gated behind a flag** so it cannot fire accidentally.

In [1]:
import json, math, os, time, random, sys
from collections import deque
from datetime import datetime
from pathlib import Path

import numpy as np
import polars as pl
import cohere
from cohere.core.api_error import ApiError
from cohere.core.request_options import RequestOptions
from dotenv import load_dotenv

# --- project root, per the repo convention (walk up to ModelPipeline) -------
root = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    if p.name == "ModelPipeline":
        root = p
        break
if root is None:
    raise RuntimeError("Could not locate the ModelPipeline root from cwd")
PKG = root / "finrag_ml_tg1"
print("ModelPipeline root:", root)

# --- inputs / outputs -------------------------------------------------------
META_PATH    = PKG / "data_cache" / "meta_embeds" / "finrag_fact_sentences_meta_embeds.parquet"
VECTORS_MAIN = PKG / "data_cache" / "embeddings" / "cohere_1024d" / "finrag_embeddings_cohere_1024d.parquet"
SHARD_DIR    = PKG / "data_cache" / "embeddings" / "_bin3_shards"
BIN3_OUT     = PKG / "data_cache" / "embeddings" / "cohere_1024d_bin3" / "finrag_embeddings_cohere_1024d_bin3.parquet"
SECRETS      = PKG / ".aws_secrets" / "aws_credentials.env"

# --- pinned model identity. 1024-d is a project-wide policy, not a knob -----
MODEL_ID      = "embed-v4.0"        # native id; Bedrock's equivalent is cohere.embed-v4:0
DIMENSIONS    = 1024                # MUST be passed explicitly - the API defaults to 1536
INPUT_TYPE    = "search_document"   # must match how Bins 1-2 were embedded
TRUNCATE      = "END"               # native vocabulary (Bedrock uses "RIGHT")

# --- batching / pacing ------------------------------------------------------
MAX_TEXTS_PER_BATCH   = 96          # hard API ceiling
MAX_TOKENS_PER_SENT   = 1000        # same outlier filter as Bins 1-2
TARGET_INPUTS_PER_MIN = 1800        # 90% of the documented 2,000/min ceiling
SHARD_EVERY_N_BATCHES = 25          # ~2,400 sentences per shard
LOG_EVERY_N_BATCHES   = 20
MAX_RETRIES           = 7
TIMEOUT_SECONDS       = 60

for d in (SHARD_DIR, BIN3_OUT.parent):
    d.mkdir(parents=True, exist_ok=True)

print("shards ->", SHARD_DIR)
print("output ->", BIN3_OUT)

ModelPipeline root: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
shards -> /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline/finrag_ml_tg1/data_cache/embeddings/_bin3_shards
output -> /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline/finrag_ml_tg1/data_cache/embeddings/cohere_1024d_bin3/finrag_embeddings_cohere_1024d_bin3.parquet


## 1. Credentials — production key

The trial key is free but capped at **1,000 API calls per month**; this run needs 1,884, so it
must use a production key. Only a masked fingerprint is printed — never the key itself.

In [2]:
load_dotenv(SECRETS, override=True)

PROD_KEY_VAR = "cohere_direct_apiprod_k1"
api_key = os.getenv(PROD_KEY_VAR)
if not api_key:
    raise RuntimeError(f"{PROD_KEY_VAR} not found in {SECRETS}")

print(f"production key '{PROD_KEY_VAR}': loaded, len={len(api_key)}, fingerprint={api_key[:4]}...{api_key[-2:]}")

client = cohere.ClientV2(api_key=api_key)
print("cohere SDK:", cohere.__version__)

production key 'cohere_direct_apiprod_k1': loaded, len=53, fingerprint=cohe...2x


cohere SDK: 7.0.8


## 2. Scope — exactly which sentences

Bin 3 is `report_year` 2022-2025. From that we keep only rows the pipeline would actually send:

- drop sentences over `1000` tokens (**44 rows**) — the same outlier rule applied to Bins 1-2,
  so this is consistent, not a new exclusion
- drop anything already present in the main vectors table (**2,644 rows** of earlier ad-hoc
  test embeds)

Reading only the `sentenceID` column from the 1.5 GB vectors table, lazily.

In [3]:
already = set(
    pl.scan_parquet(VECTORS_MAIN).select("sentenceID").collect()["sentenceID"].to_list()
)
print(f"already embedded in main table : {len(already):,}")

meta = pl.scan_parquet(META_PATH).select(
    ["sentenceID", "sentence", "report_year", "sentence_token_count"]
).collect()

bin3 = meta.filter(pl.col("report_year").is_between(2022, 2025))
elig = bin3.filter(pl.col("sentence_token_count") <= MAX_TOKENS_PER_SENT)
todo = (
    elig.filter(~pl.col("sentenceID").is_in(list(already)))
        .sort("sentenceID")                     # deterministic order -> reproducible resume
        .select(["sentenceID", "sentence", "sentence_token_count"])
)

n_tok  = todo["sentence_token_count"].sum()
n_call = math.ceil(todo.height / MAX_TEXTS_PER_BATCH)
print(f"bin3 rows                      : {bin3.height:,}")
print(f"  outliers >{MAX_TOKENS_PER_SENT} tokens (skipped) : {bin3.height - elig.height:,}")
print(f"  eligible                     : {elig.height:,}")
print(f"TO EMBED                       : {todo.height:,}")
print(f"tokens                         : {n_tok:,}   -> ${n_tok/1e6*0.12:.4f}")
print(f"calls @{MAX_TEXTS_PER_BATCH}/batch                : {n_call:,}")
print(f"est. runtime @{TARGET_INPUTS_PER_MIN}/min        : {todo.height/TARGET_INPUTS_PER_MIN:.0f} min")

already embedded in main table : 433,799


bin3 rows                      : 183,536
  outliers >1000 tokens (skipped) : 44
  eligible                     : 183,492
TO EMBED                       : 180,848
tokens                         : 7,162,360   -> $0.8595
calls @96/batch                : 1,884
est. runtime @1800/min        : 100 min


## 3. Resume state — what the shards already hold

Every shard is a finished, self-contained parquet. Resuming just means reading which
`sentenceID`s they already cover and removing those from the work list. Deleting the shard
directory is always safe; it only costs re-embedding.

In [4]:
def load_shard_paths():
    return sorted(SHARD_DIR.glob("shard_*.parquet"))

def load_done_ids():
    paths = load_shard_paths()
    if not paths:
        return set(), 0
    df = pl.concat([pl.read_parquet(p) for p in paths], how="vertical")
    return set(df["sentenceID"].to_list()), len(paths)

done_ids, n_shards = load_done_ids()
remaining = todo.filter(~pl.col("sentenceID").is_in(list(done_ids))) if done_ids else todo

print(f"existing shards        : {n_shards}")
print(f"already covered        : {len(done_ids):,}")
print(f"REMAINING THIS RUN     : {remaining.height:,}")
if remaining.height:
    rt = remaining["sentence_token_count"].sum()
    print(f"  tokens               : {rt:,}  -> ${rt/1e6*0.12:.4f}")
    print(f"  calls                : {math.ceil(remaining.height/MAX_TEXTS_PER_BATCH):,}")
else:
    print("  nothing left - skip to consolidation")

existing shards        : 0
already covered        : 0
REMAINING THIS RUN     : 180,848
  tokens               : 7,162,360  -> $0.8595
  calls                : 1,884


## 4. Pacing and error handling

**Pacing** is by *inputs* per minute, not requests — Cohere's documented Embed limit is
`2,000 inputs / min`. With 96 texts per call that is only ~20.8 calls/min, so the limiter, not
the network, sets the pace.

**Error split**, carried over from the Bedrock hard-stop fix:

- `408 / 429 / 5xx` → transient, retry with exponential backoff (capped at 4 s)
- `429 / 402 / 403` whose body mentions quota, billing, month or limit → **fatal**, abort now.
  Retrying a period cap cannot succeed, and burning the ladder against it just wastes time.
- any other `4xx` → fatal (bad request won't fix itself)

The fatal match is narrow and **falls through to the transient path** if it does not hit, so a
wording change degrades to plain retrying rather than to something worse.

In [5]:
class FatalQuotaError(RuntimeError):
    '''Provider refused for a reason retrying cannot fix (quota/billing/auth).'''

_pace = deque()   # (timestamp, n_inputs) inside the trailing 60s

def pace(n_inputs: int) -> None:
    '''Block until sending n_inputs keeps us under TARGET_INPUTS_PER_MIN.'''
    while True:
        now = time.monotonic()
        while _pace and now - _pace[0][0] >= 60.0:
            _pace.popleft()
        if sum(n for _, n in _pace) + n_inputs <= TARGET_INPUTS_PER_MIN:
            _pace.append((now, n_inputs))
            return
        time.sleep(max(60.0 - (now - _pace[0][0]), 0.05))

FATAL_HINTS = ("quota", "billing", "month", "limit exceeded", "insufficient", "suspend")

def embed_batch(texts):
    '''One embed call with retry. Returns (vectors, billed_input_tokens).'''
    attempt, delay = 0, 0.5
    while attempt < MAX_RETRIES:
        pace(len(texts))
        try:
            r = client.embed(
                model=MODEL_ID,
                input_type=INPUT_TYPE,
                texts=texts,
                embedding_types=["float"],
                output_dimension=DIMENSIONS,   # else silently 1536-d
                truncate=TRUNCATE,
                request_options=RequestOptions(
                    timeout_in_seconds=TIMEOUT_SECONDS,
                    max_retries=0,             # our loop owns retry, so pacing stays honest
                ),
            )
            vecs = r.embeddings.float_          # trailing underscore
            if len(vecs) != len(texts):
                raise FatalQuotaError(f"count mismatch: got {len(vecs)}, sent {len(texts)}")
            if any(len(v) != DIMENSIONS for v in vecs):
                bad = next(len(v) for v in vecs if len(v) != DIMENSIONS)
                raise FatalQuotaError(f"dimension mismatch: got {bad}, expected {DIMENSIONS}")
            billed = r.meta.billed_units.input_tokens if (r.meta and r.meta.billed_units) else None
            return vecs, billed

        except ApiError as e:
            status = getattr(e, "status_code", None) or 500
            body = str(getattr(e, "body", "")).lower()
            if status in (402, 403) or (status == 429 and any(h in body for h in FATAL_HINTS)):
                raise FatalQuotaError(f"cohere {status}: {body}") from e
            if status in (408, 429) or status >= 500:
                attempt += 1
                if attempt >= MAX_RETRIES:
                    raise
                nap = min(delay + random.random() * 0.25, 4.0)
                print(f"    retry {attempt}/{MAX_RETRIES} (cohere {status}) in {nap:.2f}s")
                time.sleep(nap)
                delay *= 2
                continue
            raise FatalQuotaError(f"cohere {status}: {body}") from e
    raise RuntimeError("retry loop exited without returning")

print("pacing + retry ready")

pacing + retry ready


## 5. The run

Batches are 96 texts (the token ceiling of 128k is never binding at ~40 tokens/sentence). A shard
is written every 25 batches, so a crash loses at most ~2,400 sentences (~$0.01) and the next run
picks up from the shards.

**Safe to re-run.** Interrupt it, re-run it, run it tomorrow — it resumes.

In [6]:
EMBEDDING_ID = f"cohere_direct_v4_{DIMENSIONS}d_{datetime.now().strftime('%Y%m%d_%H%M')}"
print("embedding_id for this run:", EMBEDDING_ID)

def write_shard(ids, vecs, idx):
    p = SHARD_DIR / f"shard_{idx:05d}.parquet"
    pl.DataFrame({
        "sentenceID":   ids,
        "embedding_id": [EMBEDDING_ID] * len(ids),
        "embedding":    pl.Series(vecs, dtype=pl.List(pl.Float32)),
    }).write_parquet(p, compression="zstd")
    return p

if remaining.height == 0:
    print("nothing to do")
else:
    rows = remaining.select(["sentenceID", "sentence"]).to_dicts()
    total = len(rows)
    shard_idx = (max((int(p.stem.split("_")[1]) for p in load_shard_paths()), default=0)) + 1

    buf_ids, buf_vecs = [], []
    done = billed_total = batches = 0
    t0 = time.perf_counter()

    try:
        for start in range(0, total, MAX_TEXTS_PER_BATCH):
            chunk = rows[start:start + MAX_TEXTS_PER_BATCH]
            vecs, billed = embed_batch([r["sentence"] for r in chunk])

            buf_ids.extend(r["sentenceID"] for r in chunk)
            buf_vecs.extend(vecs)
            done    += len(chunk)
            batches += 1
            if billed:
                billed_total += billed

            if batches % SHARD_EVERY_N_BATCHES == 0:
                p = write_shard(buf_ids, buf_vecs, shard_idx)
                print(f"  shard -> {p.name}  ({len(buf_ids):,} rows)")
                shard_idx += 1
                buf_ids, buf_vecs = [], []

            if batches % LOG_EVERY_N_BATCHES == 0:
                el   = time.perf_counter() - t0
                rate = done / el * 60
                eta  = (total - done) / max(rate, 1) 
                print(f"  batch {batches:,}/{math.ceil(total/MAX_TEXTS_PER_BATCH):,} | "
                      f"{done:,}/{total:,} ({done/total*100:.1f}%) | "
                      f"{rate:,.0f} inputs/min | ETA {eta:.0f} min | "
                      f"billed {billed_total:,} tok (${billed_total/1e6*0.12:.4f})")

    except FatalQuotaError as e:
        if buf_ids:
            print(f"  flushing {len(buf_ids):,} paid rows before aborting")
            write_shard(buf_ids, buf_vecs, shard_idx)
            buf_ids, buf_vecs = [], []
        print("\n" + "=" * 66)
        print("FATAL - provider refused; not retrying")
        print("=" * 66)
        print(f"  {e}")
        print(f"  embedded this run: {done:,} of {total:,}")
        print(f"  shards are intact; re-run this notebook to resume")
        print("=" * 66)
        raise

    except KeyboardInterrupt:
        if buf_ids:
            print(f"\ninterrupted - flushing {len(buf_ids):,} paid rows")
            write_shard(buf_ids, buf_vecs, shard_idx)
        raise

    if buf_ids:
        p = write_shard(buf_ids, buf_vecs, shard_idx)
        print(f"  final shard -> {p.name}  ({len(buf_ids):,} rows)")

    el = time.perf_counter() - t0
    print(f"\nRUN DONE: {done:,} sentences in {batches:,} calls, {el/60:.1f} min")
    print(f"billed {billed_total:,} tokens -> ${billed_total/1e6*0.12:.4f}")

embedding_id for this run: cohere_direct_v4_1024d_20260729_0547


  batch 20/1,884 | 1,920/180,848 (1.1%) | 1,885 inputs/min | ETA 95 min | billed 97,358.0 tok ($0.0117)


  shard -> shard_00001.parquet  (2,400 rows)


  batch 40/1,884 | 3,840/180,848 (2.1%) | 1,893 inputs/min | ETA 94 min | billed 192,087.0 tok ($0.0231)


  shard -> shard_00002.parquet  (2,400 rows)


...

  shard -> shard_00074.parquet  (2,400 rows)


  batch 1,860/1,884 | 178,560/180,848 (98.7%) | 1,731 inputs/min | ETA 1 min | billed 7,441,493.0 tok ($0.8930)


  shard -> shard_00075.parquet  (2,400 rows)


  batch 1,880/1,884 | 180,480/180,848 (99.8%) | 1,732 inputs/min | ETA 0 min | billed 7,533,013.0 tok ($0.9040)


  final shard -> shard_00076.parquet  (848 rows)

RUN DONE: 180,848 sentences in 1,884 calls, 104.3 min
billed 7,555,061.0 tokens -> $0.9066


## 6. Consolidate the shards into one Bin 3 parquet

Concatenate **lazily** (`scan_parquet` + `sink_parquet`, so ~740 MB never lands in RAM),
dedupe defensively, and validate hard before this is allowed anywhere near the main
table. The schema must match `sentenceID: String, embedding_id: String, embedding: List(Float32)`
exactly, or the later merge would fail or silently coerce.

In [7]:
shard_glob = str(SHARD_DIR / "shard_*.parquet")
paths = load_shard_paths()
print(f"consolidating {len(paths)} shards -- lazily, to avoid materializing ~740 MB")

lf = pl.scan_parquet(shard_glob)

stats = lf.select(
    pl.len().alias("rows"),
    pl.col("sentenceID").n_unique().alias("uniq"),
    pl.col("embedding").list.len().min().alias("dmin"),
    pl.col("embedding").list.len().max().alias("dmax"),
).collect().row(0, named=True)

nan_rows = lf.select(
    pl.col("embedding").list.eval(pl.element().is_nan().any()).list.first().sum().alias("n")
).collect().item()

shard_ids = set(lf.select("sentenceID").collect()["sentenceID"].to_list())

checks = {
    "no duplicate sentenceID": stats["rows"] == stats["uniq"],
    "all dims == 1024":        stats["dmin"] == stats["dmax"] == DIMENSIONS,
    "no NaN":                  nan_rows == 0,
    "schema matches main":     dict(lf.collect_schema()) == dict(pl.scan_parquet(VECTORS_MAIN).collect_schema()),
    "disjoint from main":      len(shard_ids & already) == 0,
}
print(f"  rows {stats['rows']:,} | distinct {stats['uniq']:,} | dims {stats['dmin']}..{stats['dmax']} | NaN rows {nan_rows}")
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")

if not all(checks.values()):
    raise RuntimeError("validation failed - refusing to write the Bin 3 parquet")

BIN3_OUT.parent.mkdir(parents=True, exist_ok=True)
lf.unique(subset=["sentenceID"], keep="last").sink_parquet(BIN3_OUT, compression="zstd")
print(f"\nwrote {BIN3_OUT.name}  ({BIN3_OUT.stat().st_size/1048576:.1f} MB)")
print(f"  rows in file: {pl.scan_parquet(BIN3_OUT).select(pl.len()).collect().item():,}")

consolidating 76 shards -- lazily, to avoid materializing ~740 MB


  rows 180,848 | distinct 180,848 | dims 1024..1024 | NaN rows 0
  PASS  no duplicate sentenceID
  PASS  all dims == 1024
  PASS  no NaN
  PASS  schema matches main
  PASS  disjoint from main



wrote finrag_embeddings_cohere_1024d_bin3.parquet  (641.6 MB)
  rows in file: 180,848


## 7. Merge into the main vectors table — **gated**

This is the only cell that touches the 1.5 GB production table, so it is behind an explicit flag.
Set `DO_MERGE = True` and re-run **this cell only** when the checks above are green.

It streams into a temp file, validates that, backs up the old table once, then does an atomic
swap - so the 1.5 GB table is never left half-written. S3 upload stays a separate manual step,
so nothing is published by accident.

In [8]:
DO_MERGE = True   # <-- flipped 2026-07-29 after section 6 checks came back all-PASS

if not DO_MERGE:
    print("DO_MERGE is False - nothing written. Set it to True to merge.")
else:
    import shutil
    n_main = pl.scan_parquet(VECTORS_MAIN).select(pl.len()).collect().item()
    n_bin3 = pl.scan_parquet(BIN3_OUT).select(pl.len()).collect().item()
    print(f"main {n_main:,} + bin3 {n_bin3:,} = {n_main + n_bin3:,} before dedupe")

    # Stream into a TEMP file: polars cannot sink into a file it is scanning,
    # and this also keeps the old table intact until the new one is validated.
    tmp = VECTORS_MAIN.with_suffix(".parquet.tmp")
    (pl.concat([pl.scan_parquet(VECTORS_MAIN), pl.scan_parquet(BIN3_OUT)], how="vertical")
       .unique(subset=["sentenceID"], keep="last")
       .sink_parquet(tmp, compression="zstd"))

    chk = pl.scan_parquet(tmp).select(
        pl.len().alias("rows"),
        pl.col("sentenceID").n_unique().alias("uniq"),
        pl.col("embedding").list.len().min().alias("dmin"),
        pl.col("embedding").list.len().max().alias("dmax"),
    ).collect().row(0, named=True)
    print(f"merged: {chk['rows']:,} rows | {chk['uniq']:,} distinct | dims {chk['dmin']}..{chk['dmax']}")

    assert chk["rows"] == chk["uniq"],                  "duplicate sentenceID after merge"
    assert chk["dmin"] == chk["dmax"] == DIMENSIONS,    "dimension mismatch after merge"
    assert chk["rows"] >= n_main,                       "merge lost rows vs the original table"

    backup = VECTORS_MAIN.with_suffix(".parquet.premerge_bak")
    if not backup.exists():
        shutil.copy2(VECTORS_MAIN, backup)
        print(f"backed up previous table -> {backup.name}")

    tmp.replace(VECTORS_MAIN)          # atomic swap on the same filesystem
    print(f"WROTE {VECTORS_MAIN.name}  ({VECTORS_MAIN.stat().st_size/1048576:.1f} MB)")
    print("\nProceed to section 8 for validation, then section 10 uploads and verifies.")

main 433,799 + bin3 180,848 = 614,647 before dedupe


merged: 614,647 rows | 614,647 distinct | dims 1024..1024
backed up previous table -> finrag_embeddings_cohere_1024d.parquet.premerge_bak


WROTE finrag_embeddings_cohere_1024d.parquet  (2187.3 MB)

Proceed to section 8 for validation, then section 10 uploads and verifies.


## 8. Final table validation — structural integrity and corpus coverage

The merge cell above only asserts the three things that would make it *unsafe to swap the file*.
This section is the real acceptance test on the artifact that three days of work produced, and it
lives here rather than in a new notebook.

**What "correct" means, stated as testable claims:**

1. **Structural** — one row per `sentenceID`, no nulls, every vector exactly 1024-d, no `NaN`,
   no `Inf`, no degenerate/zero vectors.
2. **Coverage** — the set of embedded IDs equals the *eligible universe* derived from the meta
   table (`sentence_token_count <= 1000`), with the only permitted gap being the 140 known
   outliers. Verified as a two-way anti-join, the SQL way: nothing missing, nothing orphaned.
3. **Per-year / per-bin** — coverage is 100% in every one of the 20 years, so no year silently
   went unembedded.

The expected total is **derived from the meta table**, not hardcoded, so this test would still
catch a bad number if the corpus changed underneath it.

Memory note: the payload is ~2.5 GB of `Float32` if materialized. Every check below projects to
small columns *before* collecting, and runs on the streaming engine, so peak RAM stays low.

In [9]:
FINAL = VECTORS_MAIN
BIN_EDGES = {"bin1 (2006-2016)": (2006, 2016), "bin2 (2017-2021)": (2017, 2021), "bin3 (2022-2025)": (2022, 2025)}

print(f"validating {FINAL.name}   ({FINAL.stat().st_size / 1048576:,.1f} MB)")
print("=" * 78)

# --- one streaming pass: derive small per-row facts, drop the float payload ---
t0 = time.perf_counter()
rowwise = pl.scan_parquet(FINAL).select(
    pl.col("sentenceID"),
    pl.col("embedding_id"),
    pl.col("embedding").list.len().alias("dim"),
    pl.col("embedding").list.eval(pl.element().pow(2)).list.sum().sqrt().alias("l2"),
    pl.col("embedding").list.eval(pl.element().is_nan()).list.any().alias("has_nan"),
    pl.col("embedding").list.eval(pl.element().is_infinite()).list.any().alias("has_inf"),
).collect(engine="streaming")
print(f"scanned {rowwise.height:,} rows in {time.perf_counter() - t0:.1f}s\n")

# --- the eligible universe, derived from the meta table (not hardcoded) ------
meta_all = pl.scan_parquet(META_PATH).select(
    ["sentenceID", "report_year", "sentence_token_count"]
).collect()
eligible = meta_all.filter(pl.col("sentence_token_count") <= MAX_TOKENS_PER_SENT)
outliers = meta_all.filter(pl.col("sentence_token_count") > MAX_TOKENS_PER_SENT)
N_EXPECTED = eligible.height

print(f"meta rows                     : {meta_all.height:,}")
print(f"  eligible (<={MAX_TOKENS_PER_SENT} tokens)      : {N_EXPECTED:,}   <- expected embedded count")
print(f"  outliers (excluded by design) : {outliers.height:,}\n")

# --- two-way anti-join: nothing missing, nothing orphaned -------------------
vec_ids = rowwise.select("sentenceID")
missing = eligible.select("sentenceID", "report_year").join(vec_ids, on="sentenceID", how="anti")
orphans = vec_ids.join(meta_all.select("sentenceID"), on="sentenceID", how="anti")
outl_embedded = outliers.select("sentenceID").join(vec_ids, on="sentenceID", how="semi")

n = rowwise.height
checks = {
    f"row count == {N_EXPECTED:,} (derived)":  n == N_EXPECTED,
    "sentenceID unique":                        rowwise["sentenceID"].n_unique() == n,
    "sentenceID non-null":                      rowwise["sentenceID"].null_count() == 0,
    "embedding_id non-null":                    rowwise["embedding_id"].null_count() == 0,
    f"every vector is {DIMENSIONS}-d":          rowwise["dim"].min() == rowwise["dim"].max() == DIMENSIONS,
    "no NaN in any vector":                     not rowwise["has_nan"].any(),
    "no Inf in any vector":                     not rowwise["has_inf"].any(),
    "no zero/degenerate vectors (L2 > 0.9)":    rowwise["l2"].min() > 0.9,
    "no eligible sentence missing":             missing.height == 0,
    "no orphan vector (ID absent from meta)":   orphans.height == 0,
    "outliers correctly left unembedded":       outl_embedded.height == 0,
}

print("STRUCTURAL + COVERAGE")
print("-" * 78)
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}   {k}")
if missing.height:
    print("\n  missing by year:")
    print(missing.group_by("report_year").len().sort("report_year"))

# --- per-year and per-bin coverage -----------------------------------------
cov = (
    eligible.select("sentenceID", "report_year")
    .join(vec_ids.with_columns(pl.lit(True).alias("embedded")), on="sentenceID", how="left")
    .group_by("report_year")
    .agg(pl.len().alias("eligible"), pl.col("embedded").fill_null(False).sum().alias("embedded"))
    .with_columns((pl.col("embedded") / pl.col("eligible") * 100).round(2).alias("pct"))
    .sort("report_year")
)
print("\nPER-YEAR COVERAGE")
print("-" * 78)
with pl.Config(tbl_rows=25):
    print(cov)

bin_rows = []
for label, (lo, hi) in BIN_EDGES.items():
    s = cov.filter(pl.col("report_year").is_between(lo, hi))
    bin_rows.append({
        "bin": label,
        "eligible": int(s["eligible"].sum()),
        "embedded": int(s["embedded"].sum()),
        "pct": round(float(s["embedded"].sum()) / float(s["eligible"].sum()) * 100, 2),
    })
print("\nPER-BIN COVERAGE")
print("-" * 78)
print(pl.DataFrame(bin_rows))

all_years_full = bool((cov["pct"] == 100.0).all())
print(f"\n  {'PASS' if all_years_full else 'FAIL'}   every one of the {cov.height} years is at 100%")

SECTION8_OK = all(checks.values()) and all_years_full
print("\n" + "=" * 78)
print(f"SECTION 8: {'ALL CHECKS PASS' if SECTION8_OK else 'FAILURES PRESENT - DO NOT UPLOAD'}")
print("=" * 78)

validating finrag_embeddings_cohere_1024d.parquet   (2,187.3 MB)


scanned 614,647 rows in 1.1s

meta rows                     : 614,787
  eligible (<=1000 tokens)      : 614,647   <- expected embedded count
  outliers (excluded by design) : 140

STRUCTURAL + COVERAGE
------------------------------------------------------------------------------
  PASS   row count == 614,647 (derived)
  PASS   sentenceID unique
  PASS   sentenceID non-null
  PASS   embedding_id non-null
  PASS   every vector is 1024-d
  PASS   no NaN in any vector
  PASS   no Inf in any vector
  PASS   no zero/degenerate vectors (L2 > 0.9)
  PASS   no eligible sentence missing
  PASS   no orphan vector (ID absent from meta)
  PASS   outliers correctly left unembedded

PER-YEAR COVERAGE
------------------------------------------------------------------------------
shape: (20, 4)
┌─────────────┬──────────┬──────────┬───────┐
│ report_year ┆ eligible ┆ embedded ┆ pct   │
│ ---         ┆ ---      ┆ ---      ┆ ---   │
│ i64         ┆ u32      ┆ u32      ┆ f64   │
╞═════════════╪══════════╪

## 9. Continuity and cross-provider consistency

Section 8 proves the final table is *internally* well-formed. It does **not** prove the merge left
the 433,799 pre-existing Bin 1-2 vectors alone - a botched concat could produce a table that
passes every structural check while silently corrupting rows. That is what this section tests,
against the `.premerge_bak` copy the merge cell made.

**On the overlap that turned out not to exist.** This cell was written expecting 2,644 collided
`sentenceID`s to need resolving, because 2,644 Bin-3-year sentences *were* already embedded during
earlier ad-hoc test runs. The measured result is **0 collisions**, and that is correct: section 2
subtracted those 2,644 IDs from the work list before embedding, so they live only in the pre-merge
table and never entered the Bin 3 parquet. The two ID sets are exactly disjoint, and
`433,799 + 180,848 = 614,647` with nothing actually deduped.

That makes a caveat this section originally worried about **moot**: `unique(subset=["sentenceID"],
keep="last")` has no guaranteed row-ordering under Polars' streaming engine, so *had* there been
collisions, which copy survived would not have been contractually fixed. With zero collisions the
`unique()` call is a no-op safety net rather than a decision point. The check is kept anyway - it
costs nothing, and it is what *proved* the sets were disjoint instead of assuming it.

The 2,644 rows are still visible in the provenance table below: Bin 3 carries 180,848 rows with a
`cohere_direct_*` `embedding_id`, plus 2,644 across three older `bedrock_*` ids, together making up
the 183,492 eligible Bin 3 sentences.

**Cross-provider consistency.** Cohere v4 returns L2-normalised vectors. If Bin 3 had come back at
the wrong dimension, from the wrong model, or unnormalised, the per-bin norm distribution would
separate visibly. Identical norm statistics across a Bedrock bin and a Cohere-direct bin is a
cheap, strong signal that the two transports really did produce one coherent vector space.

In [10]:
BACKUP = FINAL.with_suffix(".parquet.premerge_bak")
SAMPLE_N = 3000
rng = np.random.default_rng(42)

if not BACKUP.exists():
    raise RuntimeError(f"{BACKUP.name} missing - cannot verify continuity. Did the merge run?")

pre_ids  = set(pl.scan_parquet(BACKUP).select("sentenceID").collect()["sentenceID"].to_list())
bin3_ids = set(pl.scan_parquet(BIN3_OUT).select("sentenceID").collect()["sentenceID"].to_list())
fin_ids  = set(rowwise["sentenceID"].to_list())
collided = pre_ids & bin3_ids

print(f"pre-merge IDs   : {len(pre_ids):,}")
print(f"bin3 IDs        : {len(bin3_ids):,}")
print(f"final IDs       : {len(fin_ids):,}")
print(f"collided (in both, resolved by unique) : {len(collided):,}")
print(f"union check     : {len(pre_ids | bin3_ids):,} == final {len(fin_ids):,}"
      f"  -> {'OK' if len(pre_ids | bin3_ids) == len(fin_ids) else 'MISMATCH'}\n")


def fetch(path, ids):
    """Pull {sentenceID: np.ndarray} for a small ID set. Streaming, so the filter is
    applied during the scan and only the sampled rows ever land in memory."""
    df = (pl.scan_parquet(path)
            .filter(pl.col("sentenceID").is_in(list(ids)))
            .select(["sentenceID", "embedding_id", "embedding"])
            .collect(engine="streaming"))
    vecs = {sid: np.asarray(v, dtype=np.float32)
            for sid, v in zip(df["sentenceID"].to_list(), df["embedding"].to_list())}
    eids = dict(zip(df["sentenceID"].to_list(), df["embedding_id"].to_list()))
    return vecs, eids


# ---- A. strict continuity on NON-collided pre-merge rows --------------------
clean = sorted(pre_ids - collided)
samp = [clean[i] for i in rng.choice(len(clean), size=min(SAMPLE_N, len(clean)), replace=False)]
pre_v, pre_e = fetch(BACKUP, samp)
fin_v, fin_e = fetch(FINAL, samp)

identical = sum(1 for s in samp if np.array_equal(pre_v[s], fin_v[s]))
eid_same  = sum(1 for s in samp if pre_e[s] == fin_e[s])
print("A. CONTINUITY - untouched Bin 1-2 rows")
print("-" * 78)
print(f"  sampled {len(samp):,} of {len(clean):,} non-collided pre-merge rows")
print(f"  bit-identical vectors : {identical:,}/{len(samp):,}")
print(f"  embedding_id preserved: {eid_same:,}/{len(samp):,}")
cont_ok = identical == len(samp) == eid_same
print(f"  {'PASS' if cont_ok else 'FAIL'}   pre-existing vectors carried through unchanged")

# ---- B. collided rows must match one source exactly, never blend ------------
print("\nB. COLLISION RESOLUTION - the overlap rows")
print("-" * 78)
coll_ok = True
if collided:
    csamp = sorted(collided)
    if len(csamp) > SAMPLE_N:
        csamp = [csamp[i] for i in rng.choice(len(csamp), size=SAMPLE_N, replace=False)]
    c_pre, _ = fetch(BACKUP, csamp)
    c_b3, _  = fetch(BIN3_OUT, csamp)
    c_fin, _ = fetch(FINAL, csamp)
    from_pre = sum(1 for s in csamp if np.array_equal(c_fin[s], c_pre[s]))
    from_b3  = sum(1 for s in csamp if np.array_equal(c_fin[s], c_b3[s]))
    neither  = sum(1 for s in csamp
                   if not np.array_equal(c_fin[s], c_pre[s]) and not np.array_equal(c_fin[s], c_b3[s]))
    cos = [float(np.dot(c_pre[s], c_b3[s])) for s in csamp]   # both unit-norm -> dot == cosine
    print(f"  sampled {len(csamp):,} of {len(collided):,} collided rows")
    print(f"  final == pre-merge (Bedrock) : {from_pre:,}")
    print(f"  final == bin3 (Cohere direct): {from_b3:,}")
    print(f"  final == NEITHER (corruption): {neither:,}")
    print(f"  agreement between the two sources: cosine mean {np.mean(cos):.5f} "
          f"min {np.min(cos):.5f} max {np.max(cos):.5f}")
    coll_ok = neither == 0
    print(f"  {'PASS' if coll_ok else 'FAIL'}   every collided row is bit-identical to one source")
else:
    print("  no collisions")

# ---- C. per-bin norm + provenance (cross-provider consistency) --------------
year_of = meta_all.select(["sentenceID", "report_year"])
bins = (rowwise.select(["sentenceID", "embedding_id", "l2", "dim"])
        .join(year_of, on="sentenceID", how="left")
        .with_columns(
            pl.when(pl.col("report_year") <= 2016).then(pl.lit("bin1 (2006-2016)"))
             .when(pl.col("report_year") <= 2021).then(pl.lit("bin2 (2017-2021)"))
             .otherwise(pl.lit("bin3 (2022-2025)")).alias("bin")))

print("\nC. PER-BIN VECTOR STATISTICS - L2 norm should be ~1.0 in every bin")
print("-" * 78)
stats = (bins.group_by("bin").agg(
            pl.len().alias("rows"),
            pl.col("l2").mean().round(6).alias("l2_mean"),
            pl.col("l2").std().alias("l2_std"),
            pl.col("l2").min().round(6).alias("l2_min"),
            pl.col("l2").max().round(6).alias("l2_max"),
            pl.col("dim").min().alias("dim"),
         ).sort("bin"))
with pl.Config(tbl_width_chars=200):
    print(stats)

norm_ok = bool(((stats["l2_mean"] - 1.0).abs() < 1e-3).all())
spread = float(stats["l2_mean"].max() - stats["l2_mean"].min())
print(f"\n  {'PASS' if norm_ok else 'FAIL'}   all bins L2-normalised (mean within 1e-3 of 1.0)")
print(f"  max spread in mean norm across bins: {spread:.2e}   <- Bedrock vs Cohere-direct agreement")

print("\nPROVENANCE - embedding_id by bin")
print("-" * 78)
with pl.Config(tbl_rows=30, fmt_str_lengths=60, tbl_width_chars=200):
    print(bins.group_by(["bin", "embedding_id"]).len().sort(["bin", "len"], descending=[False, True]))

SECTION9_OK = cont_ok and coll_ok and norm_ok
print("\n" + "=" * 78)
print(f"SECTION 9: {'ALL CHECKS PASS' if SECTION9_OK else 'FAILURES PRESENT - DO NOT UPLOAD'}")
print("=" * 78)

pre-merge IDs   : 433,799
bin3 IDs        : 180,848
final IDs       : 614,647
collided (in both, resolved by unique) : 0
union check     : 614,647 == final 614,647  -> OK



A. CONTINUITY - untouched Bin 1-2 rows
------------------------------------------------------------------------------
  sampled 3,000 of 433,799 non-collided pre-merge rows
  bit-identical vectors : 3,000/3,000
  embedding_id preserved: 3,000/3,000
  PASS   pre-existing vectors carried through unchanged

B. COLLISION RESOLUTION - the overlap rows
------------------------------------------------------------------------------
  no collisions

C. PER-BIN VECTOR STATISTICS - L2 norm should be ~1.0 in every bin
------------------------------------------------------------------------------
shape: (3, 7)
┌──────────────────┬────────┬─────────┬───────────┬──────────┬──────────┬──────┐
│ bin              ┆ rows   ┆ l2_mean ┆ l2_std    ┆ l2_min   ┆ l2_max   ┆ dim  │
│ ---              ┆ ---    ┆ ---     ┆ ---       ┆ ---      ┆ ---      ┆ ---  │
│ str              ┆ u32    ┆ f32     ┆ f32       ┆ f32      ┆ f32      ┆ u32  │
╞══════════════════╪════════╪═════════╪═══════════╪══════════╪═════════

## 10. Cloud sync — upload and byte-level verification

S3 is the source of truth for this artifact (`DataPipeline/CLOUD_SOURCE_OF_TRUTH.md`); the local
parquet is gitignored and the Seagate archive holds the pre-merge copy. So "done" means local and
cloud are provably the same bytes, not just the same filename.

**The gate.** This cell refuses to upload unless `SECTION8_OK and SECTION9_OK`. Publishing is
outward-facing and hard to walk back, so it must not run on a table that failed a check.

**How the verification actually proves equality.** A comparison of file *sizes* would miss silent
corruption, and for a multipart upload the `ETag` is not a plain MD5 — it is
`md5(concat(md5(part) for each part)) + "-N"`. That is checkable, but only if the part size is
known, so this cell pins `multipart_chunksize` to a fixed 64 MB instead of letting the transfer
manager pick adaptively. The composite ETag is then recomputed from the local file and compared to
what S3 returns.

That is a real end-to-end integrity proof: every byte of the local file participates in the digest,
and S3's value is computed independently on its side. Two further belts: `ChecksumAlgorithm=SHA256`
makes S3 validate each part in transit and reject a corrupt one, and `ContentLength` is compared
exactly. This works here because the bucket uses SSE-S3 (`AES256`) — under SSE-KMS the ETag would
not be an MD5 and this check would have to change.

In [11]:
import hashlib
import boto3
from boto3.s3.transfer import TransferConfig

BUCKET    = os.getenv("S3_BUCKET_NAME", "sentence-data-ingestion-mjs")
KEY       = f"ML_EMBED_ASSETS/EMBED_VECTORS/cohere_1024d/{FINAL.name}"
CHUNK     = 64 * 1024 * 1024        # pinned so the composite ETag is reproducible
local_sz  = FINAL.stat().st_size

if not (SECTION8_OK and SECTION9_OK):
    raise RuntimeError("sections 8/9 did not fully pass - refusing to upload")

s3 = boto3.client("s3", region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1"))


def composite_etag(path, chunk):
    """Reproduce S3's multipart ETag: md5(concat of part md5s) + '-N'. Plain md5 if single-part."""
    part_digests = []
    with open(path, "rb") as fh:
        while True:
            block = fh.read(chunk)
            if not block:
                break
            part_digests.append(hashlib.md5(block).digest())
    if len(part_digests) == 1:
        return hashlib.md5(open(path, "rb").read()).hexdigest()
    return f"{hashlib.md5(b''.join(part_digests)).hexdigest()}-{len(part_digests)}"


# --- what is up there right now ---------------------------------------------
try:
    before = s3.head_object(Bucket=BUCKET, Key=KEY)
    print(f"remote BEFORE : {before['ContentLength']:,} B   ETag {before['ETag']}   "
          f"{before['LastModified']:%Y-%m-%d %H:%M}")
except s3.exceptions.ClientError:
    print("remote BEFORE : (object does not exist)")
print(f"local         : {local_sz:,} B  ({local_sz / 1048576:,.1f} MB)\n")

# --- upload ------------------------------------------------------------------
_seen = {"n": 0, "next": CHUNK * 4}


def progress(n):
    _seen["n"] += n
    if _seen["n"] >= _seen["next"]:
        pct = _seen["n"] / local_sz * 100
        print(f"    uploaded {_seen['n'] / 1048576:,.0f} / {local_sz / 1048576:,.0f} MB ({pct:.0f}%)")
        _seen["next"] += CHUNK * 4


print(f"uploading -> s3://{BUCKET}/{KEY}")
t0 = time.perf_counter()
s3.upload_file(
    str(FINAL), BUCKET, KEY,
    Config=TransferConfig(multipart_threshold=CHUNK, multipart_chunksize=CHUNK, max_concurrency=8),
    ExtraArgs={"ChecksumAlgorithm": "SHA256"},
    Callback=progress,
)
el = time.perf_counter() - t0
print(f"upload done in {el / 60:.1f} min ({local_sz / 1048576 / el:.1f} MB/s)\n")

# --- verify ------------------------------------------------------------------
after       = s3.head_object(Bucket=BUCKET, Key=KEY, ChecksumMode="ENABLED")
remote_sz   = after["ContentLength"]
remote_etag = after["ETag"].strip('"')
local_etag  = composite_etag(FINAL, CHUNK)

sync = {
    "byte size matches exactly":        remote_sz == local_sz,
    "composite ETag matches":           remote_etag == local_etag,
    "server-side encryption present":   "ServerSideEncryption" in after,
}
print("CLOUD SYNC VERIFICATION")
print("-" * 78)
print(f"  local  size {local_sz:,} B   ETag {local_etag}")
print(f"  remote size {remote_sz:,} B   ETag {remote_etag}")
if "ChecksumSHA256" in after:
    print(f"  remote ChecksumSHA256 : {after['ChecksumSHA256']}  ({after.get('ChecksumType', 'n/a')})")
for k, v in sync.items():
    print(f"  {'PASS' if v else 'FAIL'}   {k}")

SYNC_OK = all(sync.values())
print("\n" + "=" * 78)
print(f"SECTION 10: {'LOCAL AND CLOUD ARE BYTE-IDENTICAL' if SYNC_OK else 'SYNC VERIFICATION FAILED'}")
print("=" * 78)

print("\nFINAL VERDICT")
print("-" * 78)
for name, ok in (("section 8  structural + coverage", SECTION8_OK),
                 ("section 9  continuity + provider", SECTION9_OK),
                 ("section 10 cloud sync", SYNC_OK)):
    print(f"  {'PASS' if ok else 'FAIL'}   {name}")
print(f"\n  {'READY for Stage 3 (S3 Vectors bulk insert)' if SECTION8_OK and SECTION9_OK and SYNC_OK else '  NOT READY'}")

remote BEFORE : 1,558,533,985 B   ETag "1304d6d716e66ad3e8aa09ea32556ec7-47"   2026-07-29 08:14
local         : 2,293,538,065 B  (2,187.3 MB)

uploading -> s3://sentence-data-ingestion-mjs/ML_EMBED_ASSETS/EMBED_VECTORS/cohere_1024d/finrag_embeddings_cohere_1024d.parquet


    uploaded 256 / 2,187 MB (12%)


    uploaded 512 / 2,187 MB (23%)


    uploaded 768 / 2,187 MB (35%)


    uploaded 1,024 / 2,187 MB (47%)


...

    uploaded 1,536 / 2,187 MB (70%)


    uploaded 1,792 / 2,187 MB (82%)


    uploaded 2,048 / 2,187 MB (94%)


upload done in 5.9 min (6.2 MB/s)



CLOUD SYNC VERIFICATION
------------------------------------------------------------------------------
  local  size 2,293,538,065 B   ETag b47a120c6558e28f55f3770d18f1e9fa-35
  remote size 2,293,538,065 B   ETag b47a120c6558e28f55f3770d18f1e9fa-35
  remote ChecksumSHA256 : SUw5rmuq3jHZ0UlEuvsfJM5/+5048dWVGPgaVFck4xo=-35  (COMPOSITE)
  PASS   byte size matches exactly
  PASS   composite ETag matches
  PASS   server-side encryption present

SECTION 10: LOCAL AND CLOUD ARE BYTE-IDENTICAL

FINAL VERDICT
------------------------------------------------------------------------------
  PASS   section 8  structural + coverage
  PASS   section 9  continuity + provider
  PASS   section 10 cloud sync

  READY for Stage 3 (S3 Vectors bulk insert)


## 11. Backfill Stage 2's `embedding_id` for Bin 3

Stage 2 (`finrag_fact_sentences_meta_embeds.parquet`) carries 5 bookkeeping columns -
`embedding_id`, `embedding_model`, `embedding_dims`, `embedding_date`, `embedding_ref` - that
the production Bedrock pipeline (`embedding_generation.py`) stamps as it embeds. This notebook
deliberately bypassed that pipeline (by design - a duplicated one-off script, not a change to
production code), so those 5 columns were never stamped for Bin 3's 180,848 rows even though
the vectors themselves are complete and correct in the merged table.

**Net effect if left alone:** anything that trusts Stage 2's `embedding_id` as "has this sentence
been embedded" - including `platform_core/s3vectors_table_preparation.py`'s Stage 3 join filter -
would treat all of Bin 3 as unembedded and silently drop it.

**What "backfill" means here, precisely:**
- `embedding_id` - not invented. Borrowed verbatim from the merged vectors table's own
  `embedding_id` for that `sentenceID` (e.g. `cohere_direct_v4_1024d_20260729_0547`).
- `embedding_model`, `embedding_dims`, `embedding_ref` - set to the exact fixed values every
  existing row already uses (`cohere.embed-v4:0`, `1024`, the merged table's S3 URI) - verified
  below by inspecting real Bin 1/2 rows first, not assumed.
- `embedding_date` - stamped with the time this backfill runs, matching how the existing values
  already behave (they reflect write time, not the original run's own timestamp - verified below).

**What must NOT change:**
- The 433,799 already-stamped rows (Bin 1 + 2, plus 2,644 earlier ad-hoc Bin-3 test rows) -
  byte-for-byte identical after this runs.
- The 140 corpus-wide token-length outliers - correctly remain `null` forever; they were never
  embedded, by design, and should not look embedded.

**Expected result:** non-null `embedding_id` goes from 433,799 to exactly 614,647 - matching the
vectors table's row count exactly, since every eligible sentence is now genuinely embedded.

This was dry-run tested against the real files before writing: the join arithmetic, a
byte-identical check on previously-stamped rows, and a spot check of both an untouched
pre-existing row and a freshly backfilled row all came back correct.

In [12]:
from datetime import datetime

S3_REF = f"s3://{c.bucket}/{c.embeddings_path('cohere_1024d')}" if 'c' in dir() else \
    "s3://sentence-data-ingestion-mjs/ML_EMBED_ASSETS/EMBED_VECTORS/cohere_1024d/finrag_embeddings_cohere_1024d.parquet"

print("[Backfill] Loading Stage 2 (eager - it's ~65MB, safe) and vectors table's ID+embedding_id (2 cheap columns)")
meta_before = pl.read_parquet(META_PATH)
vec_ids = (
    pl.scan_parquet(VECTORS_MAIN)
      .select(["sentenceID", "embedding_id"])
      .collect()
      .rename({"embedding_id": "vec_embedding_id"})
)

# --- verify the fixed-value convention against real existing rows, don't assume it ---
existing = meta_before.filter(pl.col("embedding_id").is_not_null())
conv_model = existing["embedding_model"].unique().to_list()
conv_dims  = existing["embedding_dims"].unique().to_list()
conv_ref   = existing["embedding_ref"].unique().to_list()
print(f"  existing embedding_model values : {conv_model}")
print(f"  existing embedding_dims values  : {conv_dims}")
print(f"  existing embedding_ref values   : {conv_ref}")
if len(conv_model) != 1 or len(conv_dims) != 1 or len(conv_ref) != 1:
    raise RuntimeError("Convention is not a single fixed value across existing rows - stopping, this needs a human look")
MODEL_CONV, DIMS_CONV, REF_CONV = conv_model[0], conv_dims[0], conv_ref[0]

n_null_before = meta_before["embedding_id"].null_count()
n_target = (
    meta_before.filter(pl.col("embedding_id").is_null())
    .join(vec_ids.select("sentenceID"), on="sentenceID", how="semi")
    .height
)
n_stays_null = n_null_before - n_target
print(f"\n  Stage 2 rows, embedding_id null       : {n_null_before:,}")
print(f"    -> present in vectors table (BACKFILL): {n_target:,}  (expect 180,848)")
print(f"    -> absent from vectors table (stays null): {n_stays_null:,}  (expect 140, the token-length outliers)")

now = datetime.now()
was_null = pl.col("embedding_id").is_null() & pl.col("vec_embedding_id").is_not_null()

meta_after = (
    meta_before.join(vec_ids, on="sentenceID", how="left")
    .with_columns([
        pl.coalesce(["embedding_id", "vec_embedding_id"]).alias("embedding_id"),
        pl.when(was_null).then(pl.lit(MODEL_CONV)).otherwise(pl.col("embedding_model")).alias("embedding_model"),
        pl.when(was_null).then(pl.lit(DIMS_CONV, dtype=pl.Int16)).otherwise(pl.col("embedding_dims")).alias("embedding_dims"),
        pl.when(was_null).then(pl.lit(now)).otherwise(pl.col("embedding_date")).alias("embedding_date"),
        pl.when(was_null).then(pl.lit(REF_CONV)).otherwise(pl.col("embedding_ref")).alias("embedding_ref"),
    ])
    .drop("vec_embedding_id")
)

# ---- validate hard before writing anything ----
print("\n[Backfill] Validating")
checks = {}
checks["schema unchanged"] = meta_after.schema == meta_before.schema
checks["row count unchanged"] = meta_after.height == meta_before.height

n_nonnull_after = meta_after["embedding_id"].drop_nulls().len()
checks["non-null == 614,647 (matches vectors table)"] = n_nonnull_after == 614_647
checks["still-null == 140 (the token outliers)"] = meta_after["embedding_id"].null_count() == 140

old_nonnull_sorted = meta_before.filter(pl.col("embedding_id").is_not_null()).sort("sentenceID")
same_rows = (
    meta_after.join(old_nonnull_sorted.select("sentenceID"), on="sentenceID", how="semi")
    .select(old_nonnull_sorted.columns)
    .sort("sentenceID")
)
checks["433,799 previously-stamped rows byte-identical"] = old_nonnull_sorted.equals(same_rows)

# every backfilled embedding_id must equal the vectors table's own value for that sentenceID (never invented)
backfilled = meta_after.join(
    meta_before.filter(pl.col("embedding_id").is_null()).select("sentenceID"), on="sentenceID", how="semi"
).join(vec_ids, on="sentenceID", how="inner")
checks["every backfilled embedding_id matches the vectors table exactly"] = bool(
    (backfilled["embedding_id"] == backfilled["vec_embedding_id"]).all()
)

for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}   {k}")
if not all(checks.values()):
    raise RuntimeError("Backfill validation failed - refusing to write Stage 2")

# ---- write: backup, then atomic swap ----
backup = META_PATH.with_suffix(".parquet.prebackfill_bak")
if not backup.exists():
    import shutil
    shutil.copy2(META_PATH, backup)
    print(f"\n  backed up pre-backfill Stage 2 -> {backup.name}")

tmp = META_PATH.with_suffix(".parquet.tmp")
meta_after.write_parquet(tmp, compression="zstd")
tmp.replace(META_PATH)
print(f"  WROTE {META_PATH.name}  ({META_PATH.stat().st_size / 1048576:.1f} MB)")

print("\n" + "=" * 70)
print(f"BACKFILL COMPLETE: {n_target:,} rows stamped, {n_stays_null:,} correctly still null,")
print(f"embedding_id non-null: {meta_before.height - n_null_before:,} -> {n_nonnull_after:,}")
print("=" * 70)

[Backfill] Loading Stage 2 (eager - it's ~65MB, safe) and vectors table's ID+embedding_id (2 cheap columns)
  existing embedding_model values : ['cohere.embed-v4:0']
  existing embedding_dims values  : [1024]
  existing embedding_ref values   : ['s3://sentence-data-ingestion-mjs/ML_EMBED_ASSETS/EMBED_VECTORS/cohere_1024d/finrag_embeddings_cohere_1024d.parquet']

  Stage 2 rows, embedding_id null       : 180,988
    -> present in vectors table (BACKFILL): 180,848  (expect 180,848)
    -> absent from vectors table (stays null): 140  (expect 140, the token-length outliers)

[Backfill] Validating


  PASS   schema unchanged
  PASS   row count unchanged
  PASS   non-null == 614,647 (matches vectors table)
  PASS   still-null == 140 (the token outliers)
  PASS   433,799 previously-stamped rows byte-identical
  PASS   every backfilled embedding_id matches the vectors table exactly

  backed up pre-backfill Stage 2 -> finrag_fact_sentences_meta_embeds.parquet.prebackfill_bak


  WROTE finrag_fact_sentences_meta_embeds.parquet  (61.8 MB)

BACKFILL COMPLETE: 180,848 rows stamped, 140 correctly still null,
embedding_id non-null: 433,799 -> 614,647


## 12. Summary - executed 2026-07-29

| | |
|---|---|
| Sentences embedded this run | **180,848** |
| Billed tokens / cost | **7,555,061 tok / $0.9066** |
| Wall time | **104.3 min** (1,884 calls, **zero retries**) |
| Shards written | 76 |
| Final Bin 3 parquet | `cohere_1024d_bin3/...bin3.parquet`, 641.6 MB |
| Merged table total | **614,647 rows / 2,293,538,065 B (2,187.3 MB)** |
| S3 object | byte-identical, ETag `b47a120c6558e28f55f3770d18f1e9fa-35` |

Stage 2's `embedding_id` bookkeeping columns were backfilled for all 180,848 Bin 3 rows in section 11 above, so Stage 2 now correctly reflects a 100% embedded corpus, not just the vectors table.

**End state reached: 614,647 / 614,647 eligible sentences embedded = 100%** - Bin 1 206,959 +
Bin 2 224,196 + Bin 3 183,492. The 140 corpus-wide outliers over 1,000 tokens (53 / 43 / 44 by bin)
stay unembedded by design. Every one of the 20 report years is at 100% coverage.

**Validation verdict** (sections 8-10, all PASS):

- structural: unique IDs, no nulls, every vector 1024-d, no NaN, no Inf, no degenerate vectors
- coverage: two-way anti-join clean - nothing missing, nothing orphaned, outliers correctly excluded
- continuity: 3,000-row sample bit-identical to `.premerge_bak`; 0 collisions; union arithmetic exact
- provider parity: L2 norm mean 1.0 in all three bins, cross-bin spread **0.00e+00**
- cloud sync: local and S3 byte-identical by size *and* recomputed composite ETag

### Cleanup now safe
The 76 shards (636 MB) and `.premerge_bak` (1.5 GB) are redundant once the above is green. The
Seagate archive at `FinSights_Backup_20260729/` holds the pre-merge table independently.

### Next
`EMBEDDINGS_VECTORS_REVIVAL_PLAN.md` **Step E** - the Stage 3 join and bulk insert into the S3
Vectors index, which is created but still empty. That gates all retrieval measurement.